# 🚀 ESP32-S3 TinyStories on Google Colab (Free Tier)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nicholaswilde/esp32-sandbox/blob/main/projects/s3-tiny-stories/s3_tiny_stories_colab.ipynb)

This notebook builds, trains, and quantizes **TinyStories language models** for on-device inference on the **ESP32-S3** microcontroller (16MB Flash, Octal PSRAM).

> **💡 Google Colab Free Tier Instructions:**
> 1. Ensure you are using the Free Tier T4 GPU runtime: Navigate to **Runtime** > **Change runtime type**.
> 2. Select **T4 GPU** under Hardware accelerator and click **Save**.

## 1. Environment & Hardware Verification

In [ ]:
# Verify GPU accelerator (T4 Free Tier)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (Free Tier)")

## 2. Setup Workspace & Dependencies

In [ ]:
import os

# Clone repository if not already present
if not os.path.exists("esp32-sandbox"):
    !git clone https://github.com/nicholaswilde/esp32-sandbox.git

%cd /content/esp32-sandbox/projects/s3-tiny-stories

# Install required Python dependencies
!pip install -q tokenizers huggingface_hub requests numpy

## 📦 Track 1: Fast INT4 Quantization of Pre-trained TinyStories 15M

This track downloads the pre-trained `karpathy/tinyllamas` `stories15M.pt` checkpoint and quantizes it to **4-bit (INT4)** with FP16 group scales (`group_size = 64`).
The output is `stories15M_q4.bin` (~8.1 MB), perfectly sized for the 14MB ESP32-S3 Flash partition at offset `0x110000`.

In [ ]:
# Export and quantize pre-trained model
%cd /content/esp32-sandbox/projects/s3-tiny-stories/pc_tools
!python export_model.py stories15M_q4.bin

# Generate tokenizer vocab header
%cd /content/esp32-sandbox/projects/s3-tiny-stories
!python pc_tools/generate_vocab.py

In [ ]:
# Verify binary file size and header
import os
bin_path = "pc_tools/stories15M_q4.bin"
size_mb = os.path.getsize(bin_path) / (1024 * 1024)
print(f"Binary generated: {bin_path} ({size_mb:.2f} MB)")

with open(bin_path, "rb") as f:
    magic = f.read(4)
    version = int.from_bytes(f.read(4), "little")
    print(f"Header check: Magic={magic}, Version={version}")

## 🧠 Track 2: Train Custom TinyLM Model from Scratch (Free T4 GPU)

Train a custom decoder-only transformer with PyTorch on the T4 GPU:
1. **Prepare Data & Tokenizer**: Downloads the TinyStories dataset slice (~300MB) and trains a BPE tokenizer (32,768 vocabulary).
2. **Train Model**: Runs micro-batched training on the GPU.
3. **Export to INT4**: Exports the trained checkpoint into the packed binary format for ESP32 inference.

In [ ]:
# Prepare dataset slice and BPE tokenizer
%cd /content/esp32-sandbox/projects/s3-tiny-stories
!python -m research.tinystories.prepare --vocab 32768

In [ ]:
# Option A: Quick Verification Run (1.5M parameters, 500 steps, ~1-2 minutes on T4 GPU)
!python -m research.tinystories.train --arm baseline --vocab 32768 --target-core 1500000 --steps 500 --seed 0 --micro-batch-size 8

# Option B: Full Training Run (15M parameters, 5000 steps, ~15-20 minutes on T4 GPU)
# Uncomment to run full training:
# !python -m research.tinystories.train --arm baseline --vocab 32768 --target-core 15000000 --steps 5000 --seed 0 --micro-batch-size 8

In [ ]:
# Export the trained checkpoint to binary
# For 1.5M test model:
!python -m research.tinystories.export --tokenizer data/tinystories/vocab-32768/tokenizer.json baseline_v32768_c1500000_s0

# For 15M full model (if trained):
# !python -m research.tinystories.export --tokenizer data/tinystories/vocab-32768/tokenizer.json baseline_v32768_c15000000_s0

## 🧪 3. Test Text Generation in Colab

In [ ]:
# Sample text generation from the trained checkpoint
!python -m research.tinystories.sample \
  --tokenizer data/tinystories/vocab-32768/tokenizer.json \
  --run runs/baseline_v32768_c1500000_s0.pt \
  --prompt "Once upon a time, there was a little robot"


## 💾 4. Download Model Artifacts for Flashing to ESP32-S3

In [ ]:
from google.colab import files
import glob

print("Select model file to download to your local machine:")

# If Track 1 model exists, download it:
if os.path.exists("pc_tools/stories15M_q4.bin"):
    print("Downloading stories15M_q4.bin...")
    files.download("pc_tools/stories15M_q4.bin")

# If Track 2 exported models exist, download them:
for model_file in glob.glob("artifacts/tinystories/*.bin"):
    print(f"Downloading {model_file}...")
    files.download(model_file)

print("\nOnce downloaded, place the file in 'projects/s3-tiny-stories/pc_tools/' and flash with:")
print("task flash-model")